# Deploy a JupyterHub for your course

**Live demo, about 15 minutes.** A whole JupyterHub for a class, from one values file: who can log in, the menu of
environments, each student's home folder, and when idle servers stop. Run each cell with **Shift+Enter**.

This notebook runs on the **Bash** kernel, so every cell is a shell command and the cells share one shell: a variable
set in one cell is there in the next. On the training hub, `kubectl` and `helm` are ready to use.

> 📘 [Deploy JupyterHub (NRP docs)](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/) · [Zero to JupyterHub](https://z2jh.jupyter.org) · [website version of this page](https://training.nrp-nautilus.io/nairr-classroom/2_custom_jupyterhub.html)

> ### ⚠️ This demo hub uses a shared password
>
> To fit in 15 minutes, anyone who knows the password in the values file can log in. A hub for a real course logs
> students in with their campus account through **CILogon** instead, and only the people on your list get in.
> That's the one change to make before the term starts; see **After the webinar → C** at the end.

## ▶ 1. Get a namespace (1 min)

Everything you run on NRP lives in a **namespace**. For today, take one of the hundred training namespaces;
you get the same one back if you run this again:

In [ ]:
NS=$(curl -s "http://nrp-claim.nrp-training.svc.cluster.local/claim?user=$JUPYTERHUB_USER")
echo $NS

<details><summary>Expected output</summary>

<pre>nrp-training-042</pre>
</details>

Not on the training hub? Use a namespace you're an admin of instead: `NS=my-course`.

## ▶ 2. One file holds every decision (3 min)

The **Zero to JupyterHub** Helm chart is the hub; `jhub-values.yaml` says how yours should behave.
**[📄 Open ../yamls/jhub-values.yaml](../yamls/jhub-values.yaml)**, or pick out the lines that matter:

In [ ]:
grep -n -E "authenticator_class|admin_users|capacity|display_name|timeout:|hosts:" ../yamls/jhub-values.yaml

| Line | What it decides |
|---|---|
| `authenticator_class: dummy` | **Who logs in**: here anyone with the demo password; for a course, `cilogon` and your class list |
| `admin_users` | **Who runs the class**: admins see every student's server and can open it to help |
| `capacity: 5Gi` | **Each student's home folder**, kept between sessions |
| `display_name` | **The menu of environments** students pick from: Python, R, Julia, PyTorch, … |
| `timeout: 3600` | **Idle servers stop** after an hour, so the CPUs and GPUs go back to the pool |
| `hosts:` | **The web address**: `jhub-<namespace>.nrp-nautilus.io`, with HTTPS already set up |

## ▶ 3. Install it (3 min)

Tell Helm where the JupyterHub chart lives:

In [ ]:
helm repo add jupyterhub https://jupyterhub.github.io/helm-chart/
helm repo update

Install it. `sed` puts your namespace into the web address, and `-f -` hands the result to Helm. It takes about two
minutes, while the hub and its proxy start:

In [ ]:
sed "s/<namespace>/$NS/" ../yamls/jhub-values.yaml | helm install jhub jupyterhub/jupyterhub --version 4.4.5 -n $NS -f - --wait --timeout 10m

<details><summary>Expected output (the top of it)</summary>

<pre>NAME: jhub
NAMESPACE: nrp-training-042
STATUS: deployed
REVISION: 1</pre>
</details>

What Helm made: the **hub** (logins, starting servers), the **proxy** (routes each student to their server),
a small volume for the hub's database, and the web address:

In [ ]:
kubectl get pods,pvc,ingress -n $NS

## ▶ 4. Log in like a student (4 min)

In [ ]:
echo https://jhub-$NS.nrp-nautilus.io

1. Open that address and log in as **`admin`**. The password is the `password:` line under `DummyAuthenticator` in
   the values file.
2. Pick **Scipy** from the menu and press **Start**. The first start on a node can take a minute or two while the
   image downloads.
3. JupyterLab opens: this is what a student sees.
4. **File ▸ Hub Control Panel ▸ Admin**: every user and their server, with buttons to start, stop, or open a
   student's server to help them.

Each student's server is a pod of its own, with its own home volume:

In [ ]:
kubectl get pods,pvc -n $NS

## ▶ 5. Change it: a course menu (2 min)

Fifteen environments is a lot for a class. `course-menu.yaml` replaces the menu with three, one of them with a GPU:

In [ ]:
cat ../yamls/course-menu.yaml

Apply just that change to the running hub. `--reuse-values` keeps everything you installed, and the new file goes on
top:

In [ ]:
helm upgrade jhub jupyterhub/jupyterhub --version 4.4.5 -n $NS --reuse-values -f ../yamls/course-menu.yaml --wait

In the hub, **Stop My Server**, then **Start My Server**: the menu now has three entries. Students who pick the GPU
one get a GPU for their session, and give it back when the server stops or sits idle.

## ▶ 6. Clean up (1 min)

A real course hub stays up all term. Today's goes away. Remove the hub first:

In [ ]:
helm uninstall jhub -n $NS

Helm leaves two things behind on purpose: students' servers that are still running, and their home volumes, so a
reinstall would find everyone's files. Remove both:

In [ ]:
kubectl delete pods,pvc -n $NS -l app=jupyterhub

Give the training namespace back (it's also released on its own after six hours):

In [ ]:
curl -s "http://nrp-claim.nrp-training.svc.cluster.local/release?user=$JUPYTERHUB_USER"

---

# After the webinar

Each of these changes a **running** hub, so run step 3 again first. Every change follows the pattern from step 5:
a small file with just the change, applied with `helm upgrade ... --reuse-values -f <file>`.

## A. Sizes, and GPUs, per menu entry

Each menu entry can set its own CPU, memory and GPU, so one hub serves the intro unit and the deep-learning unit:

```yaml
singleuser:
  profileList:
  - display_name: "Small: 2 CPUs, 4 GB"
    default: true
    kubespawner_override:
      cpu_limit: 2
      cpu_guarantee: 2
      mem_limit: 4G
      mem_guarantee: 4G
  - display_name: "PyTorch with one GPU"
    kubespawner_override:
      image_spec: quay.io/jupyter/pytorch-notebook:cuda12-2024-04-22
      extra_resource_limits:
        nvidia.com/gpu: "1"
```

Check what a running server actually got:

In [ ]:
kubectl describe pod -n $NS -l component=singleuser-server | grep -A 6 Limits

## B. A folder the whole class shares

Make one shared volume (CephFS, which many pods can mount at once), then mount it in every student's server:

In [ ]:
kubectl apply -n $NS -f ../yamls/shared-volume.yaml

In [ ]:
helm upgrade jhub jupyterhub/jupyterhub --version 4.4.5 -n $NS --reuse-values -f ../yamls/shared-folder.yaml --wait

Restart your server: `/home/shared` is there. Anything you put in it, every student sees; each student's home folder
stays private. For a real course, mount it read-only for students.

When you clean up, delete the shared volume too (Helm didn't create it, so it doesn't remove it):

In [ ]:
kubectl delete -n $NS -f ../yamls/shared-volume.yaml

## C. Real logins with CILogon

For a course, students log in with their campus account and only the people on your list get in. Register an OAuth
client with [CILogon](https://cilogon.org/oauth2/register) first (it's reviewed by hand: allow days to a week), then
replace the `hub.config` block with the one in `cilogon-jupyterhub-config.yaml`:

In [ ]:
cat ../yamls/cilogon-jupyterhub-config.yaml

Look up your campus's identity-provider ID in the [CILogon IdP list](https://cilogon.org/idplist/); don't guess it.
`allowed_users` is your class roster.

## D. Taking it home

- Ask for a namespace for the course: [nrp.ai/education](https://nrp.ai/education/) has the steps.
- Keep your values file in git; pin `--version` so a chart release never changes the hub mid-term.
- A course image with your packages: build it in [NRP GitLab](https://nrp.ai/documentation/userdocs/development/gitlab/)
  and point a menu entry's `image_spec` at it.
- Idle culling is required on NRP, and `timeout` may not exceed 21600 (6 hours).
- The whole path: [Deploy JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/) ·
  [an example values file](https://nrp.ai/documentation/userdocs/jupyter/values/) · help in
  [NRP's chat](https://nrp.ai/contact/)